# National leakage ablation: era-leaky features vs the shipped era-free build

Reproduces, at full national scale and for **all three model families**, the C-index
inflation caused by the two era-leaky feature constructions that the pipeline retired:

1. **Per-year climate** joined at the kept row's `SURVEY_YEAR` (retired in favor of
   per-cell 1992–2025 normals). Option A keeps the *first-event* row for failures
   (median kept year 1996) but the *last observed* row for censored bridges (median
   2025), so per-year climate values fingerprint the observation era and leak
   censoring status.
2. **Kept-row reconstruction covariates** (`SURVEY_YEAR − YEAR_RECONSTRUCTED_106`,
   retired in favor of panel-entry measurement). The kept-row `YEAR_RECONSTRUCTED_106`
   is truncated at the kept year, so it separates events from censored bridges
   mechanically.

**Design** — one arm per matrix, three models, identical rows and identical seed-42
80/20 split:

| Arm | Climate | Reconstruction | RSF / Cox / AFT numbers |
|---|---|---|---|
| era-free (reference) | 1992–2025 normals | panel-entry | imported from `us_rsf_metrics.json` / `us_parametric_metrics.json` — **not refit** (this notebook's matrices are byte-identical to the shipped ones outside the swapped columns) |
| leaky | per-year at kept-row `SURVEY_YEAR` | kept-row anchored | fit here at full shipped fidelity |

The leaky arm is constructed by **column swap**: load the shipped Notebook 4 matrices,
replace only the 17 climate columns + `ever_reconstructed` +
`years_since_reconstruction` (19 columns) with their leaky variants derived from
`us_rsf_data_a.csv` + `us_climate_by_cell_year.csv`. Everything else — rows, split,
hyperparameters, imputation — matches the shipped runs exactly, so the deltas isolate
the leak.

**Re-runnability:** every heavy stage saves into `us_leakage_ablation.json`
immediately (the PA-study pattern), so a kernel restart never loses a fitted arm.
The PA case study measured this mechanism at +0.09–0.13 on single-state tree models
(`../Lu+Guler_comparison/pa_deck_climate_study.ipynb`); this notebook is the national,
three-family analogue. The README's leakage-audit numbers come from the JSON this
notebook writes.


In [1]:
# ── Configuration + era-free references ──────────────────────────────────────
import json
import time
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter, WeibullAFTFitter

DATA_CLEAN = Path("us_rsf_data_clean.csv")          # shipped RSF matrix (era-free)
DATA_PARAM = Path("us_parametric_data_clean.csv")   # shipped parametric matrix (era-free)
DATA_A     = Path("us_rsf_data_a.csv")              # kept-row SURVEY_YEAR / YEAR_RECONSTRUCTED + grid cell
CLIMATE    = Path("us_climate_by_cell_year.csv")    # per (cell, year) climate — the leaky join source
RSF_METRICS  = Path("us_rsf_metrics.json")
PARA_METRICS = Path("us_parametric_metrics.json")
OUT_JSON   = Path("us_leakage_ablation.json")

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]
SMOKE_TEST = False   # True -> _smoke inputs/outputs + tiny forest

# Copied from the era-free runs so the only difference between arms is the swapped
# columns. low_memory=True here (only risk scores are needed, not survival curves).
RSF_PARAMS = dict(n_estimators=200, max_samples=0.10, min_samples_split=100,
                  min_samples_leaf=50, max_features="sqrt", low_memory=True,
                  n_jobs=-1, random_state=42)
PENALIZER = 0.01                       # train_national_parametric.ipynb
AFT_MAXITER = 2000

if SMOKE_TEST:
    if Path("us_rsf_data_clean_smoke.csv").exists():
        DATA_CLEAN   = Path("us_rsf_data_clean_smoke.csv")
        DATA_PARAM   = Path("us_parametric_data_clean_smoke.csv")
        DATA_A       = Path("us_rsf_data_a_smoke.csv")
        RSF_METRICS  = Path("us_rsf_metrics_smoke.json")
        PARA_METRICS = Path("us_parametric_metrics_smoke.json")
    OUT_JSON = OUT_JSON.with_stem(OUT_JSON.stem + "_smoke")
    RSF_PARAMS.update(n_estimators=25, max_samples=0.5)
    print(f"SMOKE_TEST: {DATA_CLEAN} -> {OUT_JSON}")

def save_stage(key, value):
    """Persist each heavy stage to OUT_JSON immediately (resume-safe after a restart)."""
    data = json.loads(OUT_JSON.read_text()) if OUT_JSON.exists() else {}
    data[key] = value
    data["generated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    OUT_JSON.write_text(json.dumps(data, indent=2))

rsf_ref  = json.loads(RSF_METRICS.read_text())
para_ref = json.loads(PARA_METRICS.read_text())
era_free = {
    "rsf_c_index": rsf_ref["c_index_test"],
    "cox_c_index": para_ref["cox_ph"]["c_index_test"],
    "aft_c_index": para_ref["weibull_aft"]["c_index_test"],
    "n_test": rsf_ref["n_test"],
    "source": [str(RSF_METRICS), str(PARA_METRICS)],
    "note": "shipped era-free models, NOT refit here — same matrices, same seed-42 split",
}
assert rsf_ref["n_test"] == para_ref["n_test"], "era-free runs disagree on the test split"
save_stage("era_free_reference", era_free)
save_stage("design", {
    "leaky_climate": "per-year climate joined on (grid_lat, grid_lon, kept-row SURVEY_YEAR)",
    "leaky_reconstruction": "kept-row SURVEY_YEAR - kept-row YEAR_RECONSTRUCTED_106 (retired definition)",
    "rsf_params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                   for k, v in RSF_PARAMS.items()},
    "penalizer": PENALIZER,
    "split": "seed-42 80/20, identical rows to the shipped matrices",
})
print(f"era-free references: RSF {era_free['rsf_c_index']:.4f}   "
      f"Cox {era_free['cox_c_index']:.4f}   AFT {era_free['aft_c_index']:.4f}   "
      f"n_test {era_free['n_test']:,}")


era-free references: RSF 0.7547   Cox 0.8835   AFT 0.8839   n_test 194,781


In [ ]:
# ── Leaky per-bridge lookup: per-year climate + kept-row reconstruction ───────
# us_rsf_data_a.csv keeps the kept row's SURVEY_YEAR, YEAR_RECONSTRUCTED_106, and grid cell.
a = pd.read_csv(DATA_A,
                usecols=KEYS + ["SURVEY_YEAR", "YEAR_RECONSTRUCTED_106", "grid_lat", "grid_lon"],
                dtype={k: str for k in KEYS}, low_memory=False)
for k in KEYS:
    a[k] = a[k].str.strip()
assert not a.duplicated(KEYS).any(), "data_a is not one row per bridge"
a["SURVEY_YEAR"] = pd.to_numeric(a["SURVEY_YEAR"], errors="coerce").astype("int32")

CLIMATE_FEATURES = [c for c in pd.read_csv(CLIMATE, nrows=0).columns
                    if c not in ("SURVEY_YEAR", "grid_lat", "grid_lon")]
_dtypes = {c: "float32" for c in CLIMATE_FEATURES}
_dtypes.update({"SURVEY_YEAR": "int32", "grid_lat": "float64", "grid_lon": "float64"})
climate = pd.read_csv(CLIMATE, dtype=_dtypes)

# THE LEAK, arm 1: climate joined at the kept row's survey year, not as normals.
leaky = a.merge(climate, on=["grid_lat", "grid_lon", "SURVEY_YEAR"], how="left",
                validate="m:1")
del climate, a
gc.collect()

# THE LEAK, arm 2: reconstruction anchored to the kept row (the retired definition).
yr = pd.to_numeric(leaky["YEAR_RECONSTRUCTED_106"], errors="coerce")
yr = yr.mask((yr < 1600) | (yr > 2025))
leaky["ever_reconstructed_leaky"] = yr.notna().astype("int8")
ysr = leaky["SURVEY_YEAR"] - yr
leaky["years_since_reconstruction_leaky"] = ysr.mask((ysr < 0) | (ysr > 200))

SWAP_COLS = CLIMATE_FEATURES + ["ever_reconstructed", "years_since_reconstruction"]
leaky = leaky[KEYS + CLIMATE_FEATURES
              + ["ever_reconstructed_leaky", "years_since_reconstruction_leaky"]]
print(f"leaky lookup: {len(leaky):,} bridges, {len(SWAP_COLS)} columns to swap "
      f"({len(CLIMATE_FEATURES)} climate + 2 reconstruction)")
print(f"per-year climate NaN share: {leaky[CLIMATE_FEATURES].isna().mean().mean()*100:.2f}%")


def build_leaky_matrix(path):
    """Shipped era-free matrix with ONLY the SWAP_COLS replaced by leaky variants."""
    cols = pd.read_csv(path, nrows=0).columns
    dtypes = {c: "float32" for c in cols}
    dtypes.update({k: str for k in KEYS})
    dtypes["event"] = "int8"
    m = pd.read_csv(path, dtype=dtypes)
    for k in KEYS:
        m[k] = m[k].str.strip()

    present = [c for c in SWAP_COLS if c in m.columns]
    assert len(present) == len(SWAP_COLS), \
        f"matrix is missing swap columns: {set(SWAP_COLS) - set(present)}"
    merged = m[KEYS].merge(leaky, on=KEYS, how="left", validate="m:1")
    assert len(merged) == len(m), "leaky lookup changed the row count"

    nan_before = float(m[CLIMATE_FEATURES].isna().mean().mean())
    for c in CLIMATE_FEATURES:
        m[c] = merged[c].to_numpy(dtype="float32")
    m["ever_reconstructed"] = merged["ever_reconstructed_leaky"].to_numpy(dtype="float32")
    m["years_since_reconstruction"] = \
        merged["years_since_reconstruction_leaky"].to_numpy(dtype="float32")
    nan_after = float(m[CLIMATE_FEATURES].isna().mean().mean())

    # Guard: per-year coverage must track the normals', or the delta is a missingness artifact.
    print(f"{path}: swapped {len(present)} columns; "
          f"climate NaN {nan_before*100:.2f}% -> {nan_after*100:.2f}%")
    assert abs(nan_after - nan_before) < 0.02, \
        "per-year climate coverage differs from normals by >2pp — delta would be confounded"
    return m


leaky lookup: 973,906 bridges, 19 columns to swap (17 climate + 2 reconstruction)
per-year climate NaN share: 1.19%


In [3]:
# ── Leaky arm: RSF at full shipped fidelity (~85 min at national scale) ───────
m = build_leaky_matrix(DATA_CLEAN)
y = Surv.from_arrays(event=m["event"].astype(bool).to_numpy(),
                     time=m["time"].astype("float64").to_numpy())
X = m.drop(columns=["event", "time", "bridge_age"] + KEYS)   # bridge_age == time
del m
gc.collect()

# Same n + same seed -> train_test_split reproduces the shipped split membership exactly.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
assert len(X_test) == era_free["n_test"], \
    "test size differs from the era-free run — matrices are out of sync, rebuild first"

t0 = time.time()
rsf = RandomSurvivalForest(**RSF_PARAMS)
rsf.fit(X_train, y_train)
risk = np.concatenate([rsf.predict(X_test.iloc[i:i + 50_000])
                       for i in range(0, len(X_test), 50_000)])
ci_rsf_leaky = concordance_index_censored(y_test["event"], y_test["time"], risk)[0]
fit_min = (time.time() - t0) / 60

delta_rsf = ci_rsf_leaky - era_free["rsf_c_index"]
print(f"leaky RSF: C-index {ci_rsf_leaky:.4f}   era-free {era_free['rsf_c_index']:.4f}   "
      f"delta {delta_rsf:+.4f}   ({fit_min:.1f} min)")
save_stage("leaky_rsf", {
    "c_index_test": round(float(ci_rsf_leaky), 4),
    "delta_vs_era_free": round(float(delta_rsf), 4),
    "n_test": int(len(X_test)),
    "fit_minutes": round(fit_min, 1),
})

del rsf, X, X_train, X_test, y_train, y_test, risk
gc.collect()


us_rsf_data_clean.csv: swapped 19 columns; climate NaN 1.19% -> 1.19%


leaky RSF: C-index 0.7757   era-free 0.7547   delta +0.0210   (48.7 min)


129

In [4]:
# ── Leaky arm: Cox + Weibull AFT (same code path as train_national_parametric) ──
m = build_leaky_matrix(DATA_PARAM)
y = Surv.from_arrays(event=m["event"].astype(bool).to_numpy(),
                     time=m["time"].astype("float64").to_numpy())
X = m.drop(columns=["event", "time", "bridge_age"] + KEYS)
del m
gc.collect()

# Missingness flags + split + train-median imputation — same code path as Notebook 6.
na_cols = X.columns[X.isna().any()].tolist()
never_reconstructed = (X["ever_reconstructed"] == 0).to_numpy().astype("int8")
seen, flags = set(), {}
for c in na_cols:
    msk = X[c].isna().to_numpy()
    key = msk.tobytes()
    if key in seen:
        continue
    seen.add(key)
    f = msk.astype("int8")
    if (f == never_reconstructed).all():
        continue
    flags[c + "_missing"] = f
X = pd.concat([X, pd.DataFrame(flags, index=X.index)], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
assert len(X_test) == era_free["n_test"]
med = X_train[na_cols].median()
X_train, X_test = X_train.fillna(med), X_test.fillna(med)
const = X_train.columns[(X_train.max(axis=0) == X_train.min(axis=0)).to_numpy()].tolist()
if const:
    X_train, X_test = X_train.drop(columns=const), X_test.drop(columns=const)

# Cox
train_df = X_train.assign(time=y_train["time"], event=y_train["event"].astype("int8"))
t0 = time.time()
cph = CoxPHFitter(penalizer=PENALIZER)
cph.fit(train_df, duration_col="time", event_col="event")
cox_risk = cph.predict_partial_hazard(X_test).to_numpy()
ci_cox_leaky = concordance_index_censored(y_test["event"], y_test["time"], cox_risk)[0]
delta_cox = ci_cox_leaky - era_free["cox_c_index"]
print(f"leaky Cox: C-index {ci_cox_leaky:.4f}   era-free {era_free['cox_c_index']:.4f}   "
      f"delta {delta_cox:+.4f}   ({time.time()-t0:.0f}s)")
save_stage("leaky_cox", {"c_index_test": round(float(ci_cox_leaky), 4),
                         "delta_vs_era_free": round(float(delta_cox), 4),
                         "n_test": int(len(X_test))})

# Weibull AFT (train-stat standardization of continuous features, as Notebook 6)
is_binary = ((X_train == 0) | (X_train == 1)).all()
cont = X_train.columns[~is_binary]
mu, sd = X_train[cont].mean(), X_train[cont].std()
Xtr_aft, Xte_aft = X_train.copy(), X_test.copy()
Xtr_aft[cont] = (Xtr_aft[cont] - mu) / sd
Xte_aft[cont] = (Xte_aft[cont] - mu) / sd
train_df_aft = Xtr_aft.assign(time=y_train["time"], event=y_train["event"].astype("int8"))

t0 = time.time()
aft = WeibullAFTFitter(penalizer=PENALIZER)
try:
    aft.fit(train_df_aft, duration_col="time", event_col="event",
            fit_options={"maxiter": AFT_MAXITER})
except Exception as e:   # leaky per-year columns are more collinear; retry stiffer ridge
    print(f"AFT retry at 10x penalizer after {type(e).__name__}")
    aft = WeibullAFTFitter(penalizer=10 * PENALIZER)
    aft.fit(train_df_aft, duration_col="time", event_col="event",
            fit_options={"maxiter": AFT_MAXITER})
aft_median = aft.predict_median(Xte_aft).to_numpy(dtype="float64")
ci_aft_leaky = concordance_index_censored(y_test["event"], y_test["time"], -aft_median)[0]
delta_aft = ci_aft_leaky - era_free["aft_c_index"]
print(f"leaky AFT: C-index {ci_aft_leaky:.4f}   era-free {era_free['aft_c_index']:.4f}   "
      f"delta {delta_aft:+.4f}   ({(time.time()-t0)/60:.1f} min)")
save_stage("leaky_aft", {"c_index_test": round(float(ci_aft_leaky), 4),
                         "delta_vs_era_free": round(float(delta_aft), 4),
                         "n_test": int(len(X_test))})

del X, X_train, X_test, Xtr_aft, Xte_aft, train_df, train_df_aft
gc.collect()


us_parametric_data_clean.csv: swapped 19 columns; climate NaN 1.19% -> 1.19%


C:\Users\Joshu\AppData\Local\Temp\ipykernel_23784\198107850.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df = X_train.assign(time=y_train["time"], event=y_train["event"].astype("int8"))


C:\Users\Joshu\Documents\GitHub\Bridge_project_pt2\.venv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning: Column(s) ['TOLL_020_Other'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)


leaky Cox: C-index 0.8967   era-free 0.8835   delta +0.0132   (80s)


leaky AFT: C-index 0.8994   era-free 0.8839   delta +0.0155   (5.2 min)


456

In [5]:
# ── Summary table -> us_leakage_ablation.json ─────────────────────────────────
data = json.loads(OUT_JSON.read_text())
ref = data["era_free_reference"]
rows = [("RSF",         ref["rsf_c_index"], data["leaky_rsf"]),
        ("Cox PH",      ref["cox_c_index"], data["leaky_cox"]),
        ("Weibull AFT", ref["aft_c_index"], data["leaky_aft"])]

print(f"{'model':<14} {'era-free C':>11} {'leaky C':>9} {'inflation':>10}")
deltas = {}
for name, ref_c, leaky_d in rows:
    assert leaky_d["n_test"] == ref["n_test"], f"{name}: test split mismatch"
    d = leaky_d["c_index_test"] - ref_c
    deltas[name] = round(float(d), 4)
    print(f"{name:<14} {ref_c:>11.4f} {leaky_d['c_index_test']:>9.4f} {d:>+10.4f}")

save_stage("deltas", {
    "rsf": deltas["RSF"], "cox": deltas["Cox PH"], "aft": deltas["Weibull AFT"],
    "note": ("Positive = the leaky (per-year climate + kept-row reconstruction) build "
             "inflates the test C-index; these reproducible values replace the "
             "previously asserted README numbers. Reported as computed, sign included."),
})
print(f"\nsaved {OUT_JSON}")
print("These deltas are the README 'Leakage audit' numbers "
      "(previously asserted, now reproducible).")


model           era-free C   leaky C  inflation
RSF                 0.7547    0.7757    +0.0210
Cox PH              0.8835    0.8967    +0.0132
Weibull AFT         0.8839    0.8994    +0.0155

saved us_leakage_ablation.json
These deltas are the README 'Leakage audit' numbers (previously asserted, now reproducible).
